In [1]:
%load_ext autoreload
%autoreload 2

import csv

import carla
from autonomy.config import MODEL_PATH, INFERENCE_LOG_PATH
from autonomy.carla_env import connect, hide_map_layers, spawn_vehicle, CameraRig, SpeedController
from autonomy.model import SteeringModel, get_device
from autonomy.preprocessing import crop_to_rgb

In [2]:
client, world = connect()
print(world)

World(id=2059042767194134797)


In [3]:
# same road-only map as the data collection run, so the camera images match the training data
hide_map_layers(world)

Hid 232 environment objects


In [8]:
# spawn the car at the same fixed spawn point as data collection
# set spawn_index to ON_TURN_INDEX or ON_STRAIGHT_INDEX (both defined in autonomy/config.py)
# re-running would collide with the previous car at the same spot, so remove it first
from autonomy.config import ON_TURN_INDEX, ON_STRAIGHT_INDEX

spawn_index = ON_STRAIGHT_INDEX

try:
    vehicle.destroy()
except NameError:
    pass

vehicle = spawn_vehicle(world, spawn_index=spawn_index)

Spawned vehicle.tesla.model3 (id 26) at spawn point 18: Location(x=5.626226, y=133.726013, z=0.600000)


In [9]:
# forward facing camera attached to the ego vehicle
# re-running would leave the old camera attached, so remove it first
try:
    rig.destroy()
except NameError:
    pass

rig = CameraRig(world, vehicle)

Spawned camera (id 27) attached to vehicle 26


In [10]:
# load the trained steering model
device = get_device()
model = SteeringModel.load(MODEL_PATH, device)
print(f"Loaded {MODEL_PATH} on {device}")

Loaded steering_model.pt on cuda


In [11]:
# drive with the trained model: the network picks the steering, the speed controller picks the throttle.
# every tick is logged to INFERENCE_LOG_PATH; the run ends when the car leaves the driving lane
# (or when you interrupt the kernel), and then the car brakes.
carla_map = world.get_map()
speed_controller = SpeedController(vehicle)
rig.clear()  # drop stale frames from before the run

with open(INFERENCE_LOG_PATH, "w", newline="") as f:
    log = csv.writer(f)
    log.writerow(["frame", "steering"])
    try:
        while True:
            snapshot = world.wait_for_tick()  # only update once per simulator tick

            # off track = no driving lane under the car (project_to_road=False returns None there)
            if carla_map.get_waypoint(vehicle.get_location(), project_to_road=False) is None:
                print("Car left the track")
                break

            steer = model.predict(crop_to_rgb(rig.latest()))
            throttle = speed_controller.step()
            vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=steer))

            log.writerow([snapshot.timestamp.frame, steer])
            f.flush()
            print(f"speed {speed_controller.speed:.2f}  steer {steer:+.3f}")
    except KeyboardInterrupt:
        pass
    finally:
        vehicle.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0))
        print(f"Stopped, steering log in {INFERENCE_LOG_PATH}")

speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
speed 0.00  steer -0.006
